# 03 — Gold transformations (Databricks)
Runs as the final Databricks Job task after Silver. Source OHLCV timestamps are **already correct in Silver**; no year-58,000 workaround remains. Gold dimensions now consume Silver `dim_symbol` rather than re-reading raw JSON. Historical backfills and complete rebuilds are supported through widgets; derived indicators never cross missing calendar days.

In [ ]:
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")
SILVER_TABLE = "workspace.cryptolens.silver_klines"
SILVER_SYMBOL_TABLE = "workspace.cryptolens.dim_symbol"
BRONZE_EXCHANGE_INFO_TABLE = "workspace.cryptolens.bronze_exchange_info"
GOLD_SCHEMA = "workspace.gold"
LOG_TABLE = "workspace.cryptolens.pipeline_execution_logs"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

## Parameters
Normal incremental runs use a 2-day lookback. For an older backfill, set `backfill_start` and optionally `backfill_end` (YYYY-MM-DD). To repair all historic dates or replace a previously incorrect fact table, use `full_refresh=true`, then reset it to false. These widgets do not download data or move files.

In [ ]:
dbutils.widgets.dropdown("full_refresh", "false", ["true", "false"])
dbutils.widgets.text("backfill_start", "")
dbutils.widgets.text("backfill_end", "")
dbutils.widgets.text("lookback_days", "2")
dbutils.widgets.dropdown("run_optimize", "true", ["true", "false"])

FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"
BACKFILL_START = dbutils.widgets.get("backfill_start").strip() or None
BACKFILL_END = dbutils.widgets.get("backfill_end").strip() or None
LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days"))
RUN_OPTIMIZE = dbutils.widgets.get("run_optimize") == "true"
MA_SHORT, MA_LONG, VOL_WINDOW, CORR_WINDOW, CORR_TOP_N = 5, 20, 7, 14, 15
if LOOKBACK_DAYS < 0:
    raise ValueError("lookback_days cannot be negative")
if BACKFILL_END and not BACKFILL_START:
    raise ValueError("Set backfill_start when using backfill_end")
if BACKFILL_START:
    from datetime import date
    date.fromisoformat(BACKFILL_START)
    if BACKFILL_END:
        date.fromisoformat(BACKFILL_END)
        if BACKFILL_END < BACKFILL_START:
            raise ValueError("backfill_end must be after or equal to backfill_start")

FACT = f"{GOLD_SCHEMA}.fact_daily_ohlcv"
DIM_SYMBOL = f"{GOLD_SCHEMA}.dim_symbol"
DIM_DATE = f"{GOLD_SCHEMA}.dim_date"
VOL = f"{GOLD_SCHEMA}.volatility_daily"
MA = f"{GOLD_SCHEMA}.moving_averages"
CORR = f"{GOLD_SCHEMA}.symbol_correlation"
LOG_SCHEMA = spark.table(LOG_TABLE).schema
RUN_LOAD_TYPE = ("full" if FULL_REFRESH or not spark.catalog.tableExists(FACT) else "incremental")
RUN_ID = str(uuid.uuid4())
print("Gold run:", RUN_ID, "mode:", RUN_LOAD_TYPE)

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def merge_metrics(table):
    info = DeltaTable.forName(spark, table).history(1).select("operation", "operationMetrics").first()
    if info is None or info["operation"] != "MERGE":
        return None, None
    metrics = info["operationMetrics"] or {}
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def audit(table, operation, read, written, started, status, error=None, inserted=None, updated=None):
    row = [(str(uuid.uuid4()), "Silver-to-Gold",
            f"{table} [{operation}] source_rows={read} run_id={RUN_ID}",
            RUN_LOAD_TYPE, started, utc_now(), status,
            int(written if inserted is None else inserted), int(updated or 0),
            str(error)[:500] if error else None)]
    spark.createDataFrame(row, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)


@contextmanager
def tracked(table, operation):
    started = utc_now()
    metrics = {"read": 0, "written": 0}
    try:
        yield metrics
        inserted, updated = (merge_metrics(table) if operation == "MERGE" else (None, None))
        audit(table, operation, metrics["read"], metrics["written"], started, "SUCCESS", inserted=inserted, updated=updated)
        print(f"[OK] {table} [{operation}]: read={metrics['read']:,}, written={metrics['written']:,}")
    except Exception as exc:
        audit(table, operation, metrics["read"], metrics["written"], started, "FAILURE", error=exc)
        raise

## Read cleaned Silver records
Validate timestamps rather than silently correcting erroneous dates. If this fails on your existing tables, execute the one-time repair procedure from the run guide before rerunning Gold.

In [ ]:
if not spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError("Missing Silver OHLCV table; run Bronze and Silver first")
if not spark.catalog.tableExists(SILVER_SYMBOL_TABLE):
    raise RuntimeError("Missing Silver dim_symbol; ingest exchangeInfo and run Silver first")

silver_all = (spark.table(SILVER_TABLE)
    .filter(F.col("interval") == "1m")
    .select("symbol", "open_time", "open", "high", "low", "close", "volume",
            F.col("quote_asset_volume").alias("quote_volume"),
            F.col("number_of_trades").alias("trades")))
invalid_dates = silver_all.filter(
    F.col("open_time").isNull() |
    (F.col("open_time") < F.lit("2010-01-01").cast("timestamp")) |
    (F.col("open_time") >= F.lit("2100-01-01").cast("timestamp")))
if invalid_dates.limit(1).count():
    raise ValueError("Incorrect Silver open_time values detected; repair Silver timestamps before Gold")
silver_all = silver_all.withColumn("trade_date", F.to_date("open_time"))
if not silver_all.limit(1).count():
    raise ValueError("No 1-minute Silver candles available")

silver = silver_all
if not FULL_REFRESH and spark.catalog.tableExists(FACT):
    if BACKFILL_START:
        silver = silver.filter(F.col("trade_date") >= F.to_date(F.lit(BACKFILL_START)))
        if BACKFILL_END:
            silver = silver.filter(F.col("trade_date") <= F.to_date(F.lit(BACKFILL_END)))
        print("Backfill from", BACKFILL_START, "through", BACKFILL_END or "all later dates")
    else:
        last = spark.table(FACT).agg(F.max("trade_date").alias("last")).first()["last"]
        if last:
            silver = silver.filter(F.col("trade_date") >= F.date_sub(F.lit(last), LOOKBACK_DAYS))
            print(f"Incremental Gold processing from {last} minus {LOOKBACK_DAYS} days")
else:
    print("Full Gold fact rebuild from all Silver candles")

## Gold dim_symbol (from cleaned Silver)
Status history is updated on actual status changes. Initial snapshot values have a NULL `status_changed_date` and `previous_status`: being first observed does not mean the status *changed* on that date.

In [ ]:
def current_snapshot_date():
    if not spark.catalog.tableExists(BRONZE_EXCHANGE_INFO_TABLE):
        return utc_now().date().isoformat()
    row = (spark.table(BRONZE_EXCHANGE_INFO_TABLE)
        .select(F.regexp_extract("source_file", r"exchangeInfo-(\d{4}-\d{2}-\d{2})\.json", 1).alias("d"))
        .filter(F.length("d") == 10)
        .agg(F.max("d").alias("d")).first())
    return row["d"] if row and row["d"] else utc_now().date().isoformat()

SNAPSHOT_DATE = current_snapshot_date()
with tracked(DIM_SYMBOL, "MERGE") as m:
    src = (spark.table(SILVER_SYMBOL_TABLE)
        .filter(F.col("quote_asset") == "USDT")
        .select(F.abs(F.xxhash64("symbol")).alias("symbol_key"), "symbol", "base_asset", "quote_asset",
                F.when(~F.col("is_active"), F.lit("DELISTED")).otherwise(F.col("status")).alias("status"),
                "is_spot_trading_allowed")
        .withColumn("snapshot_date", F.to_date(F.lit(SNAPSHOT_DATE))))
    m["read"] = src.count()
    if not m["read"]:
        raise ValueError("Silver dim_symbol has no USDT symbols")
    if not spark.catalog.tableExists(DIM_SYMBOL):
        (src.withColumn("previous_status", F.lit(None).cast("string"))
            .withColumn("status_changed_date", F.lit(None).cast("date"))
            .withColumn("first_seen_date", F.col("snapshot_date"))
            .withColumn("last_seen_date", F.col("snapshot_date"))
            .drop("snapshot_date").write.format("delta").saveAsTable(DIM_SYMBOL))
        m["written"] = m["read"]
    else:
        (DeltaTable.forName(spark, DIM_SYMBOL).alias("t")
            .merge(src.alias("s"), "t.symbol_key = s.symbol_key")
            .whenMatchedUpdate(condition="NOT (t.status <=> s.status)", set={
                "previous_status": "t.status", "status": "s.status",
                "status_changed_date": "s.snapshot_date", "last_seen_date": "s.snapshot_date",
                "base_asset": "s.base_asset", "quote_asset": "s.quote_asset",
                "is_spot_trading_allowed": "s.is_spot_trading_allowed"})
            .whenMatchedUpdate(condition=(
                "NOT (t.base_asset <=> s.base_asset AND t.quote_asset <=> s.quote_asset AND "
                "t.is_spot_trading_allowed <=> s.is_spot_trading_allowed) OR "
                "t.last_seen_date < s.snapshot_date"), set={
                "base_asset": "s.base_asset", "quote_asset": "s.quote_asset",
                "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                "last_seen_date": "s.snapshot_date"})
            .whenNotMatchedInsert(values={
                "symbol_key": "s.symbol_key", "symbol": "s.symbol",
                "base_asset": "s.base_asset", "quote_asset": "s.quote_asset",
                "status": "s.status", "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                "previous_status": "null", "status_changed_date": "null",
                "first_seen_date": "s.snapshot_date", "last_seen_date": "s.snapshot_date"})
            .whenNotMatchedBySourceUpdate(condition="t.status <> 'DELISTED'", set={
                "previous_status": "t.status", "status": "'DELISTED'",
                "status_changed_date": f"DATE'{SNAPSHOT_DATE}'"})
            .execute())
        m["written"] = m["read"]

## Daily fact aggregation
`full_refresh=true` replaces the fact table, removing stale historical keys; otherwise use an idempotent MERGE. Existing identical business values are not rewritten just to change `_loaded_at`.

In [ ]:
with tracked(FACT, "OVERWRITE" if FULL_REFRESH else "MERGE") as m:
    m["read"] = silver.count()
    if not m["read"]:
        raise ValueError("Selected Gold date range has no Silver rows")
    daily = (silver.groupBy("symbol", "trade_date").agg(
        F.min_by("open", "open_time").alias("open_price"),
        F.max("high").alias("high_price"), F.min("low").alias("low_price"),
        F.max_by("close", "open_time").alias("close_price"),
        F.sum("volume").alias("volume"), F.sum("quote_volume").alias("quote_volume"),
        F.sum("trades").alias("trade_count"), F.count("*").alias("candle_count"))
        .withColumn("symbol_key", F.abs(F.xxhash64("symbol")))
        .withColumn("date_key", F.date_format("trade_date", "yyyyMMdd").cast("int"))
        .withColumn("vwap", F.expr("try_divide(quote_volume, volume)"))
        .withColumn("_loaded_at", F.current_timestamp())
        .select("symbol_key", "date_key", "symbol", "trade_date", "open_price", "high_price",
                "low_price", "close_price", "volume", "quote_volume", "vwap",
                "trade_count", "candle_count", "_loaded_at"))
    m["written"] = daily.count()
    if FULL_REFRESH or not spark.catalog.tableExists(FACT):
        (daily.write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(FACT))
    else:
        cols = ["symbol", "trade_date", "open_price", "high_price", "low_price",
                "close_price", "volume", "quote_volume", "vwap", "trade_count", "candle_count"]
        changed = "NOT (" + " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in cols) + ")"
        (DeltaTable.forName(spark, FACT).alias("t")
            .merge(daily.alias("s"), "t.symbol_key = s.symbol_key AND t.date_key = s.date_key")
            .whenMatchedUpdateAll(condition=changed).whenNotMatchedInsertAll().execute())

## Date dimension
Includes calendar days even when no trading data was loaded; those absent days must not be treated as adjacent trading observations.

In [ ]:
with tracked(DIM_DATE, "OVERWRITE") as m:
    bounds = spark.table(FACT).agg(F.min("trade_date"), F.max("trade_date")).first()
    dates = spark.sql(f"SELECT explode(sequence(DATE'{bounds[0]}', DATE'{bounds[1]}', interval 1 day)) AS full_date")
    dim_date = dates.select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
        F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"), F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day"), F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"))
    m["read"] = m["written"] = dim_date.count()
    dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_DATE)

## Volatility (calendar-contiguous returns)
A daily return is NULL when the previous available candle date is not yesterday. A 7-day volatility requires 7 consecutive valid daily returns (8 consecutive close dates). This prevents August 31 and September 25 from being treated as adjacent.

In [ ]:
with tracked(VOL, "OVERWRITE") as m:
    fact = spark.table(FACT)
    m["read"] = fact.count()
    w = Window.partitionBy("symbol_key").orderBy("trade_date")
    wv = w.rowsBetween(-(VOL_WINDOW - 1), 0)
    v = (fact.withColumn("prev_close", F.lag("close_price").over(w))
        .withColumn("prev_date", F.lag("trade_date").over(w))
        .withColumn("consecutive", F.datediff("trade_date", "prev_date") == 1)
        .withColumn("daily_return", F.when(F.col("consecutive"),
                                           F.expr("try_divide(close_price, prev_close) - 1")))
        .withColumn("log_return", F.when(F.col("consecutive"),
                                         F.expr("ln(try_divide(close_price, prev_close))")))
        .withColumn("intraday_range_pct", F.expr("try_divide(high_price - low_price, open_price)"))
        .withColumn("window_start", F.lag("trade_date", VOL_WINDOW).over(w))
        .withColumn("rolling_vol", F.when(
            (F.datediff("trade_date", "window_start") == VOL_WINDOW) &
            (F.count("log_return").over(wv) == VOL_WINDOW),
            F.stddev("log_return").over(wv)))
        .withColumn("rolling_vol_annualized", F.col("rolling_vol") * F.sqrt(F.lit(365.0)))
        .select("symbol_key", "date_key", "symbol", "trade_date", "close_price",
                "daily_return", "log_return", "intraday_range_pct", "rolling_vol", "rolling_vol_annualized"))
    m["written"] = v.count()
    v.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(VOL)

## Moving averages and crossover signals
Short/long averages require *consecutive calendar dates* in the observation window. If a gap interrupts the window, the average (and therefore trend/crossover) is NULL until enough new consecutive days are loaded.

In [ ]:
with tracked(MA, "OVERWRITE") as m:
    fact = spark.table(FACT)
    m["read"] = fact.count()
    w = Window.partitionBy("symbol_key").orderBy("trade_date")
    ws = w.rowsBetween(-(MA_SHORT - 1), 0)
    wl = w.rowsBetween(-(MA_LONG - 1), 0)
    series = (fact
        .withColumn("short_begin", F.lag("trade_date", MA_SHORT - 1).over(w))
        .withColumn("long_begin", F.lag("trade_date", MA_LONG - 1).over(w))
        .withColumn("ma_short", F.when(
            F.datediff("trade_date", "short_begin") == MA_SHORT - 1,
            F.avg("close_price").over(ws)))
        .withColumn("ma_long", F.when(
            F.datediff("trade_date", "long_begin") == MA_LONG - 1,
            F.avg("close_price").over(wl)))
        .withColumn("prev_short", F.lag("ma_short").over(w))
        .withColumn("prev_long", F.lag("ma_long").over(w))
        .withColumn("cross_signal",
            F.when((F.col("prev_short") <= F.col("prev_long")) &
                   (F.col("ma_short") > F.col("ma_long")), "GOLDEN_CROSS")
             .when((F.col("prev_short") >= F.col("prev_long")) &
                   (F.col("ma_short") < F.col("ma_long")), "DEATH_CROSS"))
        .withColumn("trend", F.when(F.col("ma_short") > F.col("ma_long"), "BULLISH")
                               .when(F.col("ma_short") < F.col("ma_long"), "BEARISH"))
        .select("symbol_key", "date_key", "symbol", "trade_date", "close_price",
                "ma_short", "ma_long", "cross_signal", "trend")
        .withColumn("ma_short_window", F.lit(MA_SHORT))
        .withColumn("ma_long_window", F.lit(MA_LONG)))
    m["written"] = series.count()
    series.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(MA)

## Pairwise correlation
For each of the top 15 symbols, a 14-day correlation requires 14 *adjacent calendar days* of valid returns for both symbols. Missing dates no longer get silently compressed into rolling windows.

In [ ]:
with tracked(CORR, "OVERWRITE") as m:
    returns = spark.table(VOL).filter(F.col("log_return").isNotNull())
    top = (spark.table(FACT).groupBy("symbol").agg(F.sum("quote_volume").alias("v"))
           .orderBy(F.desc("v")).limit(CORR_TOP_N))
    r = returns.join(top.select("symbol"), "symbol").select("symbol", "trade_date", "log_return")
    m["read"] = r.count()
    a, b = r.alias("a"), r.alias("b")
    paired = (a.join(b, (F.col("a.trade_date") == F.col("b.trade_date")) &
                     (F.col("a.symbol") < F.col("b.symbol")))
        .select(F.col("a.symbol").alias("symbol_a"),
                F.col("b.symbol").alias("symbol_b"),
                F.col("a.trade_date").alias("trade_date"),
                F.col("a.log_return").alias("ret_a"),
                F.col("b.log_return").alias("ret_b")))
    w = Window.partitionBy("symbol_a", "symbol_b").orderBy("trade_date")
    wr = w.rowsBetween(-(CORR_WINDOW - 1), 0)
    corr = (paired
        .withColumn("window_start", F.lag("trade_date", CORR_WINDOW - 1).over(w))
        .withColumn("rolling_corr", F.corr("ret_a", "ret_b").over(wr))
        .withColumn("obs", F.count("ret_a").over(wr))
        .filter((F.col("obs") == CORR_WINDOW) &
                (F.datediff("trade_date", "window_start") == CORR_WINDOW - 1))
        .withColumn("symbol_a_key", F.abs(F.xxhash64("symbol_a")))
        .withColumn("symbol_b_key", F.abs(F.xxhash64("symbol_b")))
        .withColumn("date_key", F.date_format("trade_date", "yyyyMMdd").cast("int"))
        .select("symbol_a_key", "symbol_b_key", "date_key", "symbol_a", "symbol_b",
                "trade_date", "rolling_corr"))
    m["written"] = corr.count()
    corr.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CORR)

## Optimization and final checks
`OPTIMIZE ... ZORDER` can be disabled for inexpensive demo runs via the widget, but remains available as requested by Phase 3.

In [ ]:
if RUN_OPTIMIZE:
    for table, cols in [(FACT, "symbol_key, date_key"), (VOL, "symbol_key, date_key"),
                        (MA, "symbol_key, date_key"), (CORR, "symbol_a_key, date_key")]:
        with tracked(table, "OPTIMIZE") as m:
            spark.sql(f"OPTIMIZE {table} ZORDER BY ({cols})")

for name in [DIM_SYMBOL, DIM_DATE, FACT, VOL, MA, CORR]:
    print(f"{name}: {spark.table(name).count():,}")
orphans = spark.table(FACT).join(spark.table(DIM_SYMBOL), "symbol_key", "left_anti").count()
print("Fact rows lacking Gold dim_symbol:", orphans)
if orphans:
    raise ValueError(f"Gold contains {orphans} orphan fact rows")
display(spark.table(LOG_TABLE).filter(F.col("layer") == "Silver-to-Gold")
        .orderBy(F.desc("start_time")).limit(15))